<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 강화학습·에이전트 · 04. Prioritized Replay

## Prioritized Experience Replay

- **원 논문:** [Prioritized Experience Replay](https://arxiv.org/abs/1511.05952)
- **저자 / 발표:** Tom Schaul, John Quan, Ioannis Antonoglou, and David Silver · ICLR (2016)
- **난이도 / 예상 시간:** 중급 · 약 55분
- **선수 지식:** DQN, TD error, importance sampling
- **로컬 학습 데이터:** `data/field_curriculum/gridworld.json`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** TD-error proportional sampling과 annealed importance weights로 로컬 transition Q-table을 학습한다.

**축소하거나 생략한 부분:** 논문은 Atari replay trees와 Double DQN을 사용한다. 축소판은 모든 transition을 메모리에 두고 정확한 categorical probabilities를 계산한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.2 | TD-error priority | 큰 surprise를 가진 transition을 더 자주 replay한다. |
| §3.3, Eq. (1) | proportional sampling | P(i)=p_i^alpha/sum p^alpha와 p=|delta|+epsilon을 구현한다. |
| §3.4 and Algorithm 1 | importance weights and prioritized Double DQN | w_i=(N P(i))^-beta를 정규화해 weighted TD loss를 만든다. |

## 핵심 재현

        Eq. (1)로 TD error가 큰 전이를 자주 뽑되, §3.4의 importance weight로 바뀐 sampling
        distribution의 편향을 완화합니다.

In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(441)
np.random.seed(441)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# Tiny scalar/table loops may be faster on CPU even when a GPU exists;
# AI_LAB_DEVICE=cpu remains the low-overhead override.
DATA_PATH = Path("data/field_curriculum/gridworld.json")
assert DATA_PATH.exists(), f"준비된 로컬 데이터가 없습니다: {DATA_PATH}"
config = json.loads(DATA_PATH.read_text(encoding="utf-8"))
required = {
    "seed",
    "size",
    "start_state",
    "terminal_state",
    "gamma",
    "transitions",
    "bandit_means",
    "offline_action_noise",
}
assert required.issubset(config), f"JSON 키 불일치: {sorted(config)}"
rows = config["transitions"]
states = torch.tensor([int(r["state"]) for r in rows], dtype=torch.long)
actions = torch.tensor([int(r["action"]) for r in rows], dtype=torch.long)
rewards = torch.tensor([float(r["reward"]) for r in rows], dtype=torch.float32)
next_states = torch.tensor([int(r["next_state"]) for r in rows], dtype=torch.long)
dones = torch.tensor([float(r["done"]) for r in rows], dtype=torch.float32)
n_states = int(max(states.max(), next_states.max()).item()) + 1
n_actions = int(actions.max().item()) + 1
gamma = float(config["gamma"])
start_state, terminal_state = int(config["start_state"]), int(config["terminal_state"])
transition_table = {
    (int(r["state"]), int(r["action"])): (
        int(r["next_state"]),
        float(r["reward"]),
        bool(r["done"]),
    )
    for r in rows
}


def encode(s):
    return F.one_hot(torch.as_tensor(s, dtype=torch.long), n_states).float()


def encode_device(s):
    return encode(s).to(DEVICE)


def env_step(state, action):
    return transition_table[(int(state), int(action))]


def value_iteration(iterations=300):
    q = torch.zeros(n_states, n_actions)
    for _ in range(iterations):
        old = q.clone()
        for (s, a), (ns, r, done) in transition_table.items():
            q[s, a] = r if done else r + gamma * old[ns].max()
    return q


q_star = value_iteration()
assert torch.isfinite(q_star).all() and len(rows) > 0
print(ACCELERATOR.summary())
print(
    "RL environment, replay metadata, and tree search stay on CPU; "
    "neural batches use DEVICE."
)
print(
    f"local MDP: states={n_states}, actions={n_actions}, "
    f"transitions={len(rows)}, gamma={gamma}"
)

## 포트폴리오 구현 설계: 수식 → 에이전트 책임 → 검증

논문의 핵심 update를 실행 가능한 에이전트의 `forward`, `loss`, `update`,
`evaluate`로 나눕니다. 아래 식의 선택·평가·gradient 경계를 메서드 본문에서
한 줄씩 추적하세요.

$$
P(i)=\frac{p_i^\alpha}{\sum_kp_k^\alpha},\qquad w_i=\left(NP(i)\right)^{-\beta}/\max_jw_j
$$

- **기호와 역할:** $p_i=|\delta_i|+\epsilon$은 priority, $\alpha$는 prioritization, $\beta$는 importance correction 강도입니다.
- **shape/state 계약:** `priority/probability/weight [N_replay], sampled indices [B], TD error [B]`
- **논문 위치:** `§3.2`의 **TD-error priority**
- **코드 Task:** 비균등 sampling, IS-weighted TD loss와 priority 갱신을 구현합니다.
- **학습·평가 흐름:** priority → categorical sample → weighted TD update → |TD| priority → 빈도 평가
- **원문 대비 한계:** 논문은 Atari replay trees와 Double DQN을 사용한다. 축소판은 모든 transition을 메모리에 두고 정확한 categorical probabilities를 계산한다.
- **구현 이유:** target, gradient, optimizer 책임을 Agent 메서드별로
  분리해 코드와 논문을 한 줄씩 대조할 수 있게 합니다.
- **완료 증거:** 마지막 셀에서 `update`, parameter 변화, 평가 metric을
  모두 `assert`합니다.

정답 클래스는 Bellman target, policy objective, search score 같은 핵심 계산을
불투명한 trainer 뒤에 감추지 않습니다. 실습본에서는 공개 API를 유지하면서 TODO를
채우고, 각 메서드의 입력 shape와 gradient가 끊기는 위치를 설명하세요.

In [ ]:
class PrioritizedReplayAgent(nn.Module):
    """Prioritized Replay의 핵심 학습·평가 경로. 입출력 계약: priority/probability/weight
    [N_replay], sampled indices [B], TD error [B]."""

    def __init__(self, q_table, optimizer, priorities, discount, alpha=0.7):
        """network, optimizer, target 또는 table 상태를 명시적으로 저장한다."""
        super().__init__()
        self.q_table = q_table
        self.optimizer = optimizer
        self.priorities = priorities
        self.discount = discount
        self.alpha = alpha

    def config(self):
        """재현 범위와 핵심 하이퍼파라미터를 dict로 반환한다."""
        return {"replay_size": len(self.priorities), "alpha": self.alpha}

    def forward(self, state, action):
        """policy/value/Q 순전파를 계산한다. shape 계약: priority/probability/weight [N_replay],
        sampled indices [B], TD error [B]."""
        return self.q_table[state, action]

    def sampling_probabilities(self):
        """priority를 정규화한 replay sampling 확률을 반환한다."""
        scaled_priority = self.priorities.clamp_min(1e-06).pow(self.alpha)
        return scaled_priority / scaled_priority.sum()

    def loss(self, indices, beta):
        """논문의 목적 함수를 미분 가능한 scalar tensor로 반환한다."""
        predicted_q = self.forward(states[indices], actions[indices])
        with torch.no_grad():
            next_q = self.q_table[next_states[indices]].max(dim=1).values
            target_q = rewards[indices]
            target_q = target_q + self.discount * (1.0 - dones[indices]) * next_q
        td_error = target_q - predicted_q
        probability = self.sampling_probabilities()[indices]
        raw_weight = (len(self.priorities) * probability).pow(-beta)
        importance_weight = raw_weight / raw_weight.max()
        objective = (importance_weight * td_error.square()).mean()
        return (objective, td_error)

    def update(self, indices, beta):
        """zero_grad, backward, step을 수행하고 유한한 float loss를 반환한다."""
        objective, td_error = self.loss(indices, beta)
        self.optimizer.zero_grad()
        objective.backward()
        self.optimizer.step()
        self.priorities[indices] = td_error.detach().abs() + 0.001
        return float(objective.detach())

    def evaluate(self, reference_q):
        """no_grad 경로에서 정책 또는 value 품질 지표 dict를 반환한다."""
        with torch.no_grad():
            mse = F.mse_loss(self.q_table, reference_q)
            probability = self.sampling_probabilities()
        return {"q_mse": float(mse), "probability_sum": float(probability.sum())}

### 단계 2. 핵심 연산 검증

- **논문의 어느 부분인가:** `§3.3, Eq. (1)` — **proportional sampling**
- **구현 이유:** P(i)=p_i^alpha/sum p^alpha와 p=|delta|+epsilon을 구현한다.
- **읽을 코드:** 아래 `implementation` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
def per_probabilities(priority, alpha=0.7):
    scaled = priority.clamp_min(1e-6).pow(alpha)
    return scaled / scaled.sum()


def is_weights(probability, beta=0.5):
    raw = (len(probability) * probability).pow(-beta)
    return raw / raw.max()


p = per_probabilities(torch.tensor([1.0, 2.0, 4.0]))
w = is_weights(p)
assert torch.allclose(p.sum(), torch.tensor(1.0)) and torch.allclose(
    w.max(), torch.tensor(1.0)
)

### 단계 3. 학습·업데이트

- **논문의 어느 부분인가:** `§3.4 and Algorithm 1` — **importance weights and prioritized Double DQN**
- **구현 이유:** w_i=(N P(i))^-beta를 정규화해 weighted TD loss를 만든다.
- **읽을 코드:** 아래 `training` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
# This tabular update is coupled to NumPy sampling, so it intentionally
# stays on CPU; neural replay variants move complete batches instead.
q = nn.Parameter(torch.zeros(n_states, n_actions))
opt = torch.optim.Adam([q], lr=0.05)
priority = torch.ones(len(rows))
rng = np.random.default_rng(442)
history = []
counts = np.zeros(len(rows), dtype=int)

portfolio_agent = PrioritizedReplayAgent(q, opt, priority, gamma)
initial_parameters = q.detach().clone()
for step in range(300):
    probability = portfolio_agent.sampling_probabilities().detach().numpy()
    index_numpy = rng.choice(
        len(rows),
        size=min(16, len(rows)),
        replace=True,
        p=probability,
    )
    counts += np.bincount(index_numpy, minlength=len(rows))
    index = torch.tensor(index_numpy)
    beta = 0.4 + 0.6 * step / 299
    history.append(portfolio_agent.update(index, beta))
parameter_delta = float((q.detach() - initial_parameters).abs().sum())
assert parameter_delta > 0.0 and np.isfinite(history).all()
assert priority.min() > 0

### 단계 4. 평가·시각화

- **논문의 어느 부분인가:** `§3.2` — **TD-error priority**
- **구현 이유:** 큰 surprise를 가진 transition을 더 자주 replay한다.
- **읽을 코드:** 아래 `evaluation` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
portfolio_metrics = portfolio_agent.evaluate(q_star)
assert portfolio_metrics

probe_priority = torch.arange(1, 7, dtype=torch.float32)
expected = per_probabilities(probe_priority).cpu().numpy()
draws = np.random.default_rng(443).choice(6, size=5000, p=expected)
empirical = np.bincount(draws, minlength=6) / 5000
corr = float(np.corrcoef(expected, empirical)[0, 1])
q_mse = float(F.mse_loss(q.detach(), q_star))
print(f"sampling corr={corr:.3f}, Q MSE={q_mse:.4f}")
assert corr > 0.95 and math.isfinite(q_mse)
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].bar(np.arange(6) - 0.18, expected, 0.36, label="Eq.(1)")
axes[0].bar(np.arange(6) + 0.18, empirical, 0.36, label="empirical")
axes[0].legend()
axes[1].plot(history)
axes[1].set_title("IS-weighted TD loss")
fig.tight_layout()
plt.show()

alpha=0은 uniform replay입니다. beta=1은 sampling 확률 변화에 대한 완전한 importance correction이며 논문은 학습 말기에 beta를 1로 anneal합니다.

### 단계 5. 통합 Agent가 실제 update와 evaluate를 소유하는지 검증

- **논문 위치:** `§3.2`의 **TD-error priority**
- **핵심 식:**

  $$
  P(i)=\frac{p_i^\alpha}{\sum_kp_k^\alpha},\qquad w_i=\left(NP(i)\right)^{-\beta}/\max_jw_j
  $$

- **입출력 shape:** `priority/probability/weight [N_replay], sampled indices [B], TD error [B]`
- **구현 이유:** 앞 셀은 target, objective, search를 작은 연산으로
  분해해 확인하고, 이 셀은 같은 구성 요소를
  `PrioritizedReplayAgent`에 주입해 최종 학습·평가 경로를
  하나로 묶습니다. 큰 surprise를 가진 transition을 더 자주 replay한다.
- **완료 증거:** `update(...)`가 유한한 loss를 반환하고,
  `parameter_delta > 0`이며, `evaluate(...)`의 정책·가치 지표가 유한해야 합니다.
  `config()`만 호출하는 것은 실행 재현으로 보지 않습니다.

In [ ]:
config_contract = portfolio_agent.config()
numeric_metrics = [
    float(value)
    for value in portfolio_metrics.values()
    if isinstance(value, (int, float))
]
assert config_contract and len(history) > 0
assert np.isfinite(np.asarray(history, dtype=float)).all()
assert parameter_delta > 0.0
assert numeric_metrics and np.isfinite(numeric_metrics).all()
print(
    {
        "class": type(portfolio_agent).__name__,
        "steps": len(history),
        "parameter_delta": parameter_delta,
        "metrics": portfolio_metrics,
    }
)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 논문은 Atari replay trees와 Double DQN을 사용한다. 축소판은 모든 transition을 메모리에 두고 정확한 categorical probabilities를 계산한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.